# Sinus Approximator

## SKaiNET dependecies

In [ ]:
USE {
    repositories {
        mavenLocal()
    }
    dependencies {
        implementation("sk.ainet.app:kotlin-notebook:0.2.0")
    }
}

### Jetbrains libs

In [ ]:
%use kandy
%use dataframe

### SKaiNET libs

In [ ]:
import sk.ainet.context.DirectCpuExecutionContext
import sk.ainet.context.ExecutionContext
import sk.ainet.context.data
import sk.ainet.execute.context.computation
import sk.ainet.lang.nn.definition
import sk.ainet.lang.nn.network
import sk.ainet.lang.model.dnn.mlp.pretrained.SinusApproximatorWandB
import sk.ainet.lang.tensor.dsl.tensor
import sk.ainet.lang.tensor.pprint
import sk.ainet.lang.tensor.relu
import sk.ainet.lang.types.FP32
import sk.ainet.lang.nn.Module as SKModule

### Init weights and biases

In [ ]:
val sinusApproximatorWandB = SinusApproximatorWandB()

### Create model

In [ ]:
fun createModel(context: ExecutionContext) = definition<FP32, Float> {
    network(context) {
        input(1, "input")  // Single input for x value

        // First hidden layer: 1 -> 16 neurons
        dense(16, "hidden-1") {
            // Weights: 16x1 matrix - explicitly defined values
            weights {
                fromArray(
                    sinusApproximatorWandB.getLayer1WandB("").weights
                )
            }
            // Bias: 16 values - explicitly defined
            bias {
                fromArray(
                    sinusApproximatorWandB.getLayer1WandB("").bias
                )
            }
            activation = { tensor -> with(tensor) { relu() } }
        }

        // Second hidden layer: 16 -> 16 neurons
        dense(16, "hidden-2") {
            // Weights: 16x16 matrix - explicitly defined values
            weights {
                fromArray(
                    sinusApproximatorWandB.getLayer2WandB("").weights
                )
            }
            // Bias: 16 values - explicitly defined
            bias {
                fromArray(
                    sinusApproximatorWandB.getLayer2WandB("").bias
                )
            }
            activation = { tensor -> with(tensor) { relu() } }
        }

        // Output layer: 16 -> 1 neuron
        dense(1, "output") {
            // Weights: 1x16 matrix - explicitly defined values
            weights {
                fromArray(
                    sinusApproximatorWandB.getLayer3WandB("").weights
                )
            }

            // Bias: single value - explicitly defined
            bias {
                fromArray(
                    sinusApproximatorWandB.getLayer3WandB("").bias
                )
            }
        }
    }
}


In [ ]:
import sk.ainet.lang.model.Model

fun sk.ainet.lang.nn.Module<FP32, Float>.calcSine(ctx: ExecutionContext, angle: Float): Float {
    val model_: sk.ainet.lang.nn.Module<FP32, kotlin.Float> = this
    return computation<Float>(ctx) {
        // Create a simple input tensor compatible with the model's expected input size (1)
        model_.forward(
            data<FP32, Float>(ctx) {
                tensor<FP32, Float>() {
                    // Using shape(1, 1) to represent a single scalar input in 2D form
                    shape(1, 1) {
                        fromArray(
                            floatArrayOf(angle)
                        )
                    }
                }
            }, ctx
        ).data[0, 0]
    }
}



## Calculations

Calculate 50 samples with Math.sin and predict with NN to compare

In [ ]:
val numSamples = 100
val maxInput = PI.toFloat() / 2f  // π/2

println("Neural Network vs Math.sin() Comparison")
println("=".repeat(50))
println("Input\t\tNetwork Output\tMath.sin()\tDifference")
println("-".repeat(50))

val ctx = DirectCpuExecutionContext()
val sineNn =  createModel(ctx)

var totalError = 0.0

for (i in 0 until numSamples) {
    // Generate input value from 0 to π/2
    val x = (i.toFloat() / (numSamples - 1)) * maxInput

    // Get network prediction
    val predicted = sineNn.calcSine(ctx, x)

    // Calculate true sin value
    val actual = sin(x.toDouble()).toFloat()

    // Calculate difference
    val difference = abs(predicted - actual)
    totalError += difference.toDouble()

    // Print comparison (every 10th sample for readability)
    if (i % 10 == 0) {
        println("%.4f\t\t%.4f\t\t%.4f\t\t%.4f".format(x, predicted, actual, difference))
    }
}

val meanAbsoluteError = totalError / numSamples
println("-".repeat(50))
println("Mean Absolute Error: %.6f".format(meanAbsoluteError))
println("Network approximation quality: ${if (meanAbsoluteError < 0.1) "Good" else "Needs improvement"}")



In [ ]:
val x_values = List(100) { index ->
    (index / (100 - 1).toFloat()) * (PI / 2)
}

val y_values = List(100) { index ->
    sin(x_values[index])
}


val y_nn_values = List(100) { index ->
    // Get network prediction
    sineNn.calcSine(ctx, x_values[index].toFloat())
}

val df = dataFrameOf(
    "x" to x_values + x_values,
    "y" to y_values + y_nn_values,
    "mode" to List(100) { "sin" } + List(100) { "nn" }
)

In [ ]:
df.plot {
    line {
        x("x")
        y("y")
        color("mode") {
            scale = categorical("sin" to Color.PURPLE, "nn" to Color.ORANGE)
        }
        width = 1.5
    }
}